# SilentBridge — train real base_model.pt

Dataset: **ISL-CSLTR** (Kaggle `drblack00/isl-csltr-indian-sign-language-dataset`) — 700 videos, 100 sentences, 7 signers. Small enough for Colab free GPU.

Run cells top to bottom. **Cell 5 (inspect) matters** — the CSV-builder cell after it assumes a folder layout; adjust it once you see the real structure printed.

Runtime: Runtime > Change runtime type > T4 GPU.

In [ ]:
# 1. Install deps not already in Colab
!pip install -q kagglehub mediapipe opencv-python-headless pandas

In [ ]:
# 2. Clone the repo — reuse its exact model code / training script / tokenizer
# so what trains here matches what the backend loads at inference time.
!git clone https://github.com/BharathWaj-K-R/SilentBridge.git
%cd SilentBridge

In [ ]:
# 3. Download the dataset
import kagglehub

dataset_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
print("downloaded to:", dataset_path)

In [ ]:
# 4. Inspect the actual folder layout before assuming anything
import os

count = 0
for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for f in files[:5]:
        print("  " * (depth + 1) + f)
    count += 1
    if count > 40:
        print("... (truncated, enough to see the pattern)")
        break

## 5. Build `ISLTranslate.csv` (uid,text) + a flat `<uid>.mp4` folder

`extract_keypoints.py` (from the repo) expects:
```
raw_videos/<uid>.mp4
labels.csv   with columns: uid, text
```
**Edit the cell below to match what cell 4 printed.** The code assumes video files are named/organized in a way that reveals which sentence they belong to (very common for this dataset: a folder per sentence, or a filename encoding the sentence id). Adjust `sentence_id_from_path()` accordingly — that's the only thing likely to need changing.

In [ ]:
import glob
import os
import shutil
import pandas as pd

RAW_VIDEOS_DIR = "data/raw_videos"
os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)

video_files = glob.glob(os.path.join(dataset_path, "**", "*.mp4"), recursive=True) \
    + glob.glob(os.path.join(dataset_path, "**", "*.MP4"), recursive=True)
print(f"found {len(video_files)} video files")

# --- ADJUST THIS if cell 4 shows a different layout ---
# Common ISL-CSLTR layout: .../<something>/<Sentence text or id>/<signer>/<clip>.mp4
# i.e. the sentence lives in the path itself. Fallback: use the parent folder
# name as the sentence text (works if that's literally the sentence/gloss).
def sentence_id_from_path(video_path: str) -> str:
    return os.path.basename(os.path.dirname(video_path))

rows = []
for i, vp in enumerate(video_files):
    uid = f"clip{i:04d}"
    text = sentence_id_from_path(vp).replace("_", " ").strip()
    if not text:
        continue
    dst = os.path.join(RAW_VIDEOS_DIR, f"{uid}.mp4")
    if not os.path.exists(dst):
        shutil.copy(vp, dst)
    rows.append({"uid": uid, "text": text})

df = pd.DataFrame(rows)
os.makedirs("data/labels", exist_ok=True)
df.to_csv("data/labels/ISLTranslate.csv", index=False)
print(df.shape)
df.head(10)

**Check `df.head(10)` above — `text` should read like real sentences/gloss labels, not garbage.** If it doesn't, fix `sentence_id_from_path()` using what cell 4 printed, then re-run cell 5.

In [ ]:
# 6. Extract MediaPipe pose+face keypoints (repo's own script — same format
# the model/dataset loader expects)
!python backend/scripts/extract_keypoints.py \
  --videos_dir data/raw_videos \
  --labels_csv data/labels/ISLTranslate.csv \
  --out_dir data/processed/isltranslate

In [ ]:
# 7. Train — small model, small dataset, keep epochs modest for free-tier GPU time.
# CTC loss, best-val-loss checkpoint kept (see train_base_model.py).
!PYTHONPATH=backend python -m app.training.train_base_model \
  --data-dir data/processed/isltranslate \
  --output backend/app/models/weights/base_model.pt \
  --epochs 15 \
  --batch-size 4

In [ ]:
# 8a. Download the two output files to your machine
from google.colab import files

files.download("backend/app/models/weights/base_model.pt")
files.download("backend/app/models/weights/base_model.vocab.json")
print("drop both into backend/app/models/weights/ in your local repo, then commit+push.")

## 8b. (optional) push straight from Colab instead
Only run this if you want to skip the manual download/upload step. It asks for a token at runtime (not stored in this notebook) — use a **fine-grained GitHub PAT scoped to just this repo**, and revoke it after.

In [ ]:
import getpass

token = getpass.getpass("GitHub token (input hidden, not saved to notebook): ")
!git config user.email "colab-trainer@users.noreply.github.com"
!git config user.name "SilentBridge Colab Trainer"
!git add -f backend/app/models/weights/base_model.pt backend/app/models/weights/base_model.vocab.json
!git commit -m "Train base_model.pt on ISL-CSLTR (real weights, replaces placeholder)"
!git push https://{token}@github.com/BharathWaj-K-R/SilentBridge.git main
del token